# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mashfiqmahi/assignment_FLyRank-AI/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1 — "What Predicts Health?" (ML Appendix, p.27). The Random Forest predicts Health Score from Average Position, Impressions, Scroll Depth, and CTR — but Health Score is defined as a weighted sum of exactly those same fields (Impressions 30pts + Position 30pts + CTR 20pts + Scroll depth 20pts, from p.5). My methodology question: is the "43% importance for Position" telling us anything about real-world performance drivers, or just re-discovering the scoring formula's own weights? The label here is a defined rule, not an observed outcome — so I'd read this table as descriptive of the formula, not as a causal driver ranking. (The paper itself flags this caveat, which I think is the right call.)

Finding 2 — "What Predicts Growth?" (ML Appendix, p.29). The logistic regression reports 71% holdout accuracy predicting growing vs. declining pages, using an 80/20 split. My methodology question: was this split random-by-page or grouped-by-brand? With only 57 brands behind 341,701 pages, a random split risks letting the model partly memorize brand-level style rather than learn a genuine growth signal — the same leakage risk I tested for in my own Week 5 clustering work. I'd want to see the 71% re-measured under a brand-grouped split before fully trusting it as a general pattern.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Results:** Random split — train silhouette 0.421, holdout 0.421, with all 47/47 clients appearing on both sides (confirmed leakage). Grouped split — train 0.419, holdout 0.496, with 0 clients leaked (confirmed honest).

Interpretation: The grouped-split holdout score is higher, not lower, than the random-split holdout score. Had contamination been inflating quality, fixing it should have decreased the score — instead it increased. We observe that the archetype clusters generalize at least as well to unseen clients as they do to training clients under this one honest split. This is a directional, decision-support finding, not a causal claim about why clusters generalize this well.

In [2]:
%pip -q install duckdb huggingface_hub pandas scikit-learn
import os, duckdb, pandas as pd, numpy as np
from google.colab import userdata
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")

# --- Recreate the exact Week-5 feature table (same query, same grain) ---
df = con.sql("""
WITH monthly AS (
    SELECT
        content_hash_id,
        ANY_VALUE(client_hash_id) AS client_hash_id,
        SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS impressions,
        SUM(gsc_clicks)      FILTER (WHERE gsc_data_available IS TRUE) AS clicks,
        SUM(gsc_sum_position) FILTER (WHERE gsc_data_available IS TRUE) AS sum_position
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY content_hash_id
)
SELECT
    m.content_hash_id,
    m.client_hash_id,
    m.impressions,
    CASE WHEN m.impressions > 0 THEN m.clicks * 100.0 / m.impressions ELSE NULL END AS ctr_pct,
    CASE WHEN m.impressions > 0 THEN m.sum_position * 1.0 / m.impressions ELSE NULL END AS avg_position,
    DATE_DIFF('day', d.content_created_date, DATE '2026-03-31') AS days_since_created,
    d.content_type
FROM monthly m
JOIN read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet') d
    ON m.content_hash_id = d.content_hash_id
""").df()

df["days_since_created"] = df["days_since_created"].clip(lower=0)

# Same filter as Week 5: keep only pages with real search visibility
df = df[df["impressions"] > 0].copy()

cluster_features = ["impressions", "ctr_pct", "avg_position", "days_since_created"]
FINAL_K = 5

def fit_and_score(train_df, holdout_df, label):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_df[cluster_features])
    X_holdout = scaler.transform(holdout_df[cluster_features])

    km = KMeans(n_clusters=FINAL_K, random_state=42, n_init=10)
    train_labels = km.fit_predict(X_train)
    holdout_labels = km.predict(X_holdout)

    train_sil = silhouette_score(X_train, train_labels)
    holdout_sil = silhouette_score(X_holdout, holdout_labels)

    print(f"--- {label} ---")
    print(f"Train:   {len(train_df):,} pages | silhouette = {train_sil:.3f}")
    print(f"Holdout: {len(holdout_df):,} pages | silhouette = {holdout_sil:.3f}\n")
    return train_sil, holdout_sil

# ============================================================
# BEFORE — naive random split (rows shuffled, ignoring client)
# ============================================================
df_train_rand, df_holdout_rand = train_test_split(df, test_size=0.3, random_state=42)
before_train_sil, before_holdout_sil = fit_and_score(df_train_rand, df_holdout_rand, "BEFORE: random split (dishonest)")

overlap_rand = set(df_train_rand["client_hash_id"]) & set(df_holdout_rand["client_hash_id"])
print(f"Clients appearing on BOTH sides of the random split: {len(overlap_rand)} "
      f"(out of {df['client_hash_id'].nunique()} total clients)\n")

# ============================================================
# AFTER — grouped split by client_hash_id (honest, same as Week 5)
# ============================================================
gss = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=42)
train_idx, holdout_idx = next(gss.split(df, groups=df["client_hash_id"]))
df_train_grp = df.iloc[train_idx].copy()
df_holdout_grp = df.iloc[holdout_idx].copy()

after_train_sil, after_holdout_sil = fit_and_score(df_train_grp, df_holdout_grp, "AFTER: grouped-by-client split (honest)")

overlap_grp = set(df_train_grp["client_hash_id"]) & set(df_holdout_grp["client_hash_id"])
print(f"Clients appearing on BOTH sides of the grouped split: {len(overlap_grp)} (should be 0)\n")

# ============================================================
# THE COMPARISON TABLE — the actual deliverable for this section
# ============================================================
comparison = pd.DataFrame({
    "split_type": ["random (before)", "grouped-by-client (after)"],
    "train_silhouette": [round(before_train_sil, 3), round(after_train_sil, 3)],
    "holdout_silhouette": [round(before_holdout_sil, 3), round(after_holdout_sil, 3)],
    "clients_leaked": [len(overlap_rand), len(overlap_grp)],
})
print(comparison.to_string(index=False))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- BEFORE: random split (dishonest) ---
Train:   123,716 pages | silhouette = 0.421
Holdout: 53,022 pages | silhouette = 0.421

Clients appearing on BOTH sides of the random split: 47 (out of 47 total clients)

--- AFTER: grouped-by-client split (honest) ---
Train:   130,904 pages | silhouette = 0.419
Holdout: 45,834 pages | silhouette = 0.496

Clients appearing on BOTH sides of the grouped split: 0 (should be 0)

               split_type  train_silhouette  holdout_silhouette  clients_leaked
          random (before)             0.421               0.421              47
grouped-by-client (after)             0.419               0.496               0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I audited my four final clustering features (`impressions`, `ctr_pct`, `avg_position`, `days_since_created`) with the checklist below. Full code and printed output is in the cell below.

**Results:**
- **Check 1 (product flags):** none of my features match FlyRank's own rule/flag naming patterns — clean.
- **Check 2 (window alignment):** all four features come from the same March 2026 partition, and `days_since_created` uses a fixed reference date, not "today" — so it can't accidentally pull in future information on re-run.
- **Check 3 (missingness):** 0% missing on all four features across all three content types — expected, since I already filter to visible pages (`impressions > 0`) before clustering, which removes the undefined CTR/position rows.
- **Check 4 (attack test):** I built a `health_score_proxy` — a mini version of FlyRank's own Health Score formula, using only my own already-included features — and added it as a 5th dimension. Silhouette went from 0.419 → 0.396 (a small **decrease**, not an increase).

**Interpretation of Check 4:** I expected the score might jump up (the "confession" pattern the leakage skill describes for supervised leakage), but it went down slightly instead. My read: `health_score_proxy` is a weighted recombination of features already in the model, so it adds a redundant, partly-collinear dimension rather than new information — and redundant dimensions can blur K-Means boundaries rather than sharpen them. This is a negative result worth reporting honestly rather than forcing into the pattern I expected: the model does not appear to be trivially gameable by adding a rule-style composite score. One caveat: the proxy used MinMax scaling while my real features used StandardScaler, so a scaling mismatch could also explain part of the shift — I would not treat a single −0.022 move as strong proof either way, just as one honest, non-alarming data point.
- **Check 5 (exclusions):** health_score/needs_ctr_fix (decision-derived, would be circular), content_updated_date (~88.5% implausibly future-dated), backlinks (structurally 100% missing for feedly articles), IDs (grouping only), content_type (context/naming only, never a model input).

In [5]:
# ============================================================
# Leakage audit on the final clustering feature set
# ============================================================
print("Features actually used for clustering:", cluster_features)
print()

# --- Check 1: none of our features are FlyRank product flags / rule outputs ---
# FlyRank's own rule system uses names like health_score, needs_ctr_fix, quick_win_tag.
# None of those should appear among our cluster_features.
suspect_names = ["health_score", "needs_", "flag", "quick_win", "score", "label", "tier"]
flagged = [f for f in cluster_features if any(s in f.lower() for s in suspect_names)]
print(f"Check 1 - feature names matching 'product flag' patterns: {flagged if flagged else 'NONE - clean'}")

# --- Check 2: window alignment - everything comes from the same snapshot month ---
print()
print("Check 2 - window alignment:")
print("All four features are aggregated from the SAME partition (month=2026-03).")
print("days_since_created is computed relative to a FIXED reference date (2026-03-31),")
print("not the query's run date - so it can't accidentally encode information from")
print("after the snapshot, no matter when this notebook is re-run.")

# --- Check 3: missingness pattern check (per content_type) ---
print()
print("Check 3 - missing-value rate per feature, by content_type:")
missing_by_type = df.groupby("content_type")[cluster_features].apply(lambda g: g.isna().mean())
print(missing_by_type.round(3))

# --- Check 4: THE ATTACK - add a deliberately leaky, rule-derived feature ---
# This mirrors Finding 1 from Section 1: FlyRank's own Health Score is built from
# Impressions + Position + CTR + Scroll depth. We build a similar proxy score from
# our OWN features and see if simply ADDING it inflates the silhouette score -
# even though it carries no genuinely new information (it's just a weighted
# re-combination of columns already in the model).
from sklearn.preprocessing import MinMaxScaler

mm = MinMaxScaler()
proxy_inputs = df_train_grp[["impressions", "avg_position", "ctr_pct"]].fillna(0)
proxy_scaled = mm.fit_transform(proxy_inputs)
# lower avg_position = better position, so we flip it before combining
health_score_proxy = (0.4 * proxy_scaled[:, 0]) + (0.4 * (1 - proxy_scaled[:, 1])) + (0.2 * proxy_scaled[:, 2])

df_train_leaky = df_train_grp.copy()
df_train_leaky["health_score_proxy"] = health_score_proxy

leaky_features = cluster_features + ["health_score_proxy"]
scaler_leaky = StandardScaler()
X_leaky = scaler_leaky.fit_transform(df_train_leaky[leaky_features])
km_leaky = KMeans(n_clusters=FINAL_K, random_state=42, n_init=10)
leaky_labels = km_leaky.fit_predict(X_leaky)
leaky_sil = silhouette_score(X_leaky, leaky_labels)

print()
print("Check 4 - attack test: adding a rule-derived 'health_score_proxy' feature")
print(f"Honest silhouette (4 real features, train side):  {after_train_sil:.3f}")
print(f"WITH leaky proxy feature added (5 features):       {leaky_sil:.3f}")
print(f"Difference: {leaky_sil - after_train_sil:+.3f}")

# --- Summary: fields deliberately excluded, and why ---
print()
print("Check 5 - fields deliberately EXCLUDED from clustering, and why:")
excluded = {
    "health_score / needs_ctr_fix / other product flags": "decision-derived outputs of an existing rule system - baseline to beat, never a feature (see Finding 1, Section 1)",
    "content_updated_date": "~88.5% implausibly future-dated in this warehouse release - not trustworthy as a feature",
    "backlinks": "structurally 100% missing for feedly-sourced articles - would inject a category signal if filled blindly",
    "content_hash_id / client_hash_id": "identifiers - used only for grouping/splitting, never as model inputs",
    "content_type": "kept as CONTEXT for naming clusters after fitting, never fed into the K-Means input matrix",
}
for k, v in excluded.items():
    print(f"  - {k}: {v}")


Features actually used for clustering: ['impressions', 'ctr_pct', 'avg_position', 'days_since_created']

Check 1 - feature names matching 'product flag' patterns: NONE - clean

Check 2 - window alignment:
All four features are aggregated from the SAME partition (month=2026-03).
days_since_created is computed relative to a FIXED reference date (2026-03-31),
not the query's run date - so it can't accidentally encode information from
after the snapshot, no matter when this notebook is re-run.

Check 3 - missing-value rate per feature, by content_type:
                    impressions  ctr_pct  avg_position  days_since_created
content_type                                                              
comparison article          0.0      0.0           0.0                 0.0
feedly article              0.0      0.0           0.0                 0.0
keyword article             0.0      0.0           0.0                 0.0

Check 4 - attack test: adding a rule-derived 'health_score_proxy' fea

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (Week 5, Section 3):** *"...clustering surfaces a large, systematic blind spot the threshold-based rule misses."*

**Why this is too bold:** the underlying numbers are real and solid (clusters 0+1, ~85% of visible pages, show 0% median CTR on page 1, while the Week 4 rule flags only ~33% of them for review) — but "surfaces...misses" reads as a settled, general fact about the rule, when really it's a pattern observed in one dataset, one snapshot, one training run.

**Rewritten:** *"In this dataset, clusters 0 and 1 (~85% of visible pages) show 0% median CTR while ranking on page 1, and the Week 4 rule flagged only ~33% of these pages for review. This suggests the threshold-based rule may be under-flagging pages with this specific impression/position/CTR profile — a pattern worth checking against a larger or more recent snapshot before treating it as a general property of the rule."*

---

**Original (Week 5, Section 3):** *"...confirming these are stable, recurring patterns rather than noise from one batch."*

**Why this is too bold:** "confirming" implies proof; I only ran one train/holdout split, not repeated validation.

**Rewritten:** *"The same five-cluster shape reappeared on a holdout set of 17 unseen clients, which suggests these patterns are not specific to one training batch — though this is based on a single train/holdout split, not repeated validation across multiple splits."*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.